# C4 — Évaluer, éprouver, communiquer

| Projet | Étape | Durée | Modules réinvestis |
|---|---|---|---|
| C — Reconnaître des chiffres manuscrits | 4 sur 4 | 4 h | M1, M5, M6, M8 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Trois méthodes sont prêtes et leurs réglages ont été choisis sur le paquet de validation. Il reste à répondre à la question du projet de façon **honnête** et à la **communiquer**.

**La sous-question de cette étape : quelle réussite peut-on promettre, avec quelle incertitude, et dans quelles conditions la promesse ne tient-elle plus ?**

**Activité** : Avant d'ouvrir le paquet de test, écrivez la méthode et le réglage que vous retenez, et pourquoi. Pourquoi est-il important de l'écrire **avant** ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_reussite_test = 0.98          # réussite de votre méthode préférée sur le test (modifiez)
pari_reussite_decale = 0.80        # réussite si les chiffres sont décalés d'un pixel vers la droite (modifiez)

## ③ Abstraire : deux sortes d'erreurs de mesure

Le taux de réussite mesuré sur le test n'est qu'une **estimation** de la réussite sur de futures images. Deux sortes d'erreurs l'affectent (module M0) :

- **L'erreur aléatoire** : avec un autre découpage au hasard, on aurait obtenu un autre nombre. On l'estime en répétant le découpage et en regardant la dispersion des résultats.
- **L'erreur systématique** : si les futures images diffèrent des images du jeu de données, la mesure se trompe toujours dans le même sens, et répéter le découpage n'y change rien. D'après la description du jeu, ces chiffres ont été écrits par **une dizaine de personnes seulement** : chaque scripteur a donc des chiffres dans le paquet d'apprentissage **et** dans le paquet de test.

**Éprouver la méthode.** Une transformation simple de l'image est aussi une transformation de son vecteur. Décaler l'image d'un pixel vers la droite revient à appliquer à $\mathbf{x}$ une matrice $T$ de taille $64 \times 64$ qui déplace les coordonnées ; ajouter un bruit revient à remplacer $\mathbf{x}$ par $\mathbf{x} + \boldsymbol{\varepsilon}$. Un lecteur humain ne voit presque pas la différence. Et nos méthodes ?

**Activité** : Pour une image dont le trait est épais de deux pixels, estimez à la main la distance entre l'image et sa copie décalée d'un pixel, et comparez-la à la distance typique entre deux chiffres différents (de l'ordre de 40 à 60).

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Ouvrir le paquet de test, une seule fois

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from chiffres import charger, decouper, afficher, matrice_confusion
sys.path.insert(0, "corrige")
from reconnaissance import Reconnaisseur

X, y = charger()
X_app, y_app, X_test, y_test = decouper(X, y, part_test=0.25, graine=0)

methodes = {
    "chiffre moyen": Reconnaisseur("moyenne"),
    "voisins (k = 1)": Reconnaisseur("voisins", k=1),
    "voisins (k = 3)": Reconnaisseur("voisins", k=3),
    "sous-espaces (k = 10)": Reconnaisseur("sous-espaces", k=10),
}
for r in methodes.values():
    r.ajuster(X_app, y_app)       # apprentissage sur tout le paquet hors test, réglages déjà choisis
bilan_test = pd.DataFrame({
    "réussite (test)": [r.reussite(X_test, y_test) for r in methodes.values()],
    "erreurs": [int(np.sum(r.predire(X_test) != y_test)) for r in methodes.values()],
    "valeurs conservées": [r.taille_memoire() for r in methodes.values()],
}, index=methodes.keys())
print(bilan_test.to_string(formatters={"réussite (test)": "{:.1%}".format}))

Dans la suite, on utilise le module `reconnaissance.py`, qui rassemble les trois méthodes des étapes précédentes. Vous écrirez le vôtre dans la partie ⑥.

### L'erreur aléatoire : répéter le découpage

In [ ]:
graines = range(20)
rng_affichage = np.random.default_rng(0)
repetitions = {nom: [] for nom in methodes}
for g in graines:
    Xa, ya, Xt, yt = decouper(X, y, part_test=0.25, graine=100 + g)
    for nom, r in methodes.items():
        repetitions[nom].append(Reconnaisseur(r.methode, r.k).ajuster(Xa, ya).reussite(Xt, yt))

fig, ax = plt.subplots(figsize=(7.5, 3.4))
for i, (nom, valeurs) in enumerate(repetitions.items()):
    ax.scatter(valeurs, np.full(len(valeurs), i) + rng_affichage.uniform(-0.12, 0.12, len(valeurs)),
               s=18, color=LIOUAI["vert"], alpha=0.7)
    ax.plot([np.mean(valeurs)] * 2, [i - 0.3, i + 0.3], color=LIOUAI["navy"], lw=2.5)
ax.set_yticks(range(len(repetitions))); ax.set_yticklabels(list(repetitions.keys()))
ax.set_xlabel(f"réussite sur le test, {len(graines)} découpages (trait : moyenne)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
for nom, valeurs in repetitions.items():
    print(f"{nom:22s} moyenne {np.mean(valeurs):.1%}, écart type {np.std(valeurs, ddof=1):.1%}, "
          f"de {min(valeurs):.1%} à {max(valeurs):.1%}")

**Activité** : Les écarts entre les méthodes sont-ils plus grands que la dispersion due au découpage ? Peut-on affirmer que les voisins font mieux que les sous-espaces ? Que le chiffre moyen fait moins bien que les deux autres ?

Indiquez vos réponses et justifiez-les.

### Éprouver : décaler et bruiter

In [ ]:
def decaler(images, dx=1, dy=0):
    # décale chaque image de dx pixels vers la droite et dy vers le bas ; ce qui sort est perdu, ce qui entre est blanc
    grilles = np.zeros((len(images), 8 + 2, 8 + 2))
    grilles[:, 1:9, 1:9] = images.reshape(-1, 8, 8)
    grilles = np.roll(grilles, (dy, dx), axis=(1, 2))
    return grilles[:, 1:9, 1:9].reshape(-1, 64)

rng = np.random.default_rng(0)
epreuves = {
    "aucune": X_test,
    "décalage 1 px à droite": decaler(X_test, 1, 0),
    "décalage 1 px en bas": decaler(X_test, 0, 1),
    "bruit (écart type 2)": np.clip(X_test + rng.normal(0, 2, X_test.shape), 0, 16),
    "bruit (écart type 4)": np.clip(X_test + rng.normal(0, 4, X_test.shape), 0, 16),
    "trait pâli (× 0,6)": 0.6 * X_test,
}
k_exemple = int(np.flatnonzero(y_test == 3)[0])
afficher(np.vstack([im[k_exemple] for im in epreuves.values()]), titres=list(epreuves.keys()), colonnes=6, taille=1.6)
plt.show()

robustesse = pd.DataFrame({nom: [r.reussite(Xe, y_test) for Xe in epreuves.values()] for nom, r in methodes.items()},
                          index=epreuves.keys())
print(robustesse.to_string(float_format="{:.1%}".format))

### Lire votre propre chiffre

Dessinez un chiffre dans la grille ci-dessous : `#` pour un pixel noir, `+` pour un gris, `.` pour un blanc. Gardez un trait épais, comme dans les données.

In [ ]:
dessin = '''
..####..
.##..##.
.....##.
....##..
...##...
..##....
.##.....
.######.
'''
valeurs = {".": 0, "+": 8, "#": 16}
lignes = [l for l in dessin.strip().splitlines()]
mon_chiffre = np.array([[valeurs[c] for c in ligne] for ligne in lignes], dtype=float).reshape(1, 64)

r = methodes["sous-espaces (k = 10)"]
residus_dessin = r.scores(mon_chiffre)[0]
afficher(mon_chiffre, titres=[f"lu : {r.predire(mon_chiffre)[0]}"], colonnes=1, taille=1.6)
plt.show()
print("Résidus par chiffre :", np.round(residus_dessin).astype(int))
print("Avec rejet (seuil 0,9) :", r.predire_avec_rejet(mon_chiffre, 0.9)[0], " (−1 = la machine hésite)")

## ⑤ Interpréter

In [ ]:
print(f"Vos paris : {pari_reussite_test:.0%} sur le test, {pari_reussite_decale:.0%} avec un décalage")
print(f"Résultats (sous-espaces) : {robustesse.loc['aucune', 'sous-espaces (k = 10)']:.1%} sur le test, "
      f"{robustesse.loc['décalage 1 px à droite', 'sous-espaces (k = 10)']:.1%} avec un décalage d'un pixel")

**Activité** :

1. **La promesse.** Rédigez en deux phrases la réussite que vous pouvez promettre, avec sa marge d'incertitude et les **conditions** dans lesquelles elle vaut (mêmes scripteurs, chiffres centrés et normalisés comme ceux du jeu).
2. **L'erreur systématique.** Pourquoi les résultats sur ce jeu sont-ils probablement **optimistes** pour de nouveaux scripteurs ? Quelle expérience faudrait-il pour la mesurer ? Quelles données manquent pour la faire ici ?
3. **La robustesse.** Un décalage d'un pixel, invisible pour un humain, fait chuter toutes les méthodes. Expliquez pourquoi à l'aide de la distance euclidienne. Quelle étape de prétraitement (avant la reconnaissance) proposeriez-vous ?
4. **Le trait pâli.** Pourquoi les sous-espaces sont-ils bien plus sensibles que les voisins à l'intensité du trait ? (Indication : que vaut le résidu d'une image pâle pour un sous-espace **affine**, qui passe par la moyenne ?) Comment le cosinus de l'étape C2 pourrait-il aider ?

Indiquez vos réponses et justifiez-les.

### La réponse à la question du projet

Peut-on reconnaître un chiffre manuscrit avec les seuls outils de l'algèbre linéaire ? Rassemblez vos résultats dans un tableau final : pour chaque méthode, l'idée en une phrase, le module du parcours qui la fonde, la réussite et sa dispersion, la mémoire, la robustesse, la possibilité d'expliquer une décision.

### Le rapport et la présentation

Rédigez un rapport de **4 à 6 pages** selon ce plan :

1. **La question** : ce que « reconnaître » veut dire, le critère de succès, le coût d'une erreur.
2. **Les données** : leur origine, leur format, et ce qu'elles ne représentent pas (peu de scripteurs, images déjà centrées).
3. **Les trois méthodes** : chacune avec une figure (chiffres moyens, voisins d'une image, chiffres propres) et la formule qui la résume. Définissez une seule fois vecteur, distance, sous-espace, projection, résidu, et gardez ensuite les mêmes mots.
4. **L'évaluation** : le protocole (apprentissage, validation, test ouvert une seule fois), l'erreur aléatoire, l'erreur systématique.
5. **Les épreuves** : décalage, bruit, intensité, et votre chiffre dessiné.
6. **La conclusion** : la promesse, ses conditions, et ce qu'il faudrait pour aller plus loin.

Pour la présentation de 10 minutes, **6 diapositives au plus**. Pour chaque idée, choisissez la représentation la plus parlante : une grille d'images vaut mieux qu'un tableau de pixels, un graphique de réussite selon $k$ vaut mieux qu'une liste de nombres.

### Travailler en équipe et relire

Si vous travaillez à plusieurs, répartissez les méthodes, puis **échangez vos modules** : chacun lance les tests de l'autre et essaie de le mettre en défaut (une image vide, un seul exemple d'un chiffre, des étiquettes manquantes). Échangez aussi vos rapports et rédigez pour l'autre trois remarques précises : une force, une affirmation insuffisamment justifiée, une figure à améliorer.

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Les limites du modèle.** Toutes nos méthodes regardent l'image comme un vecteur de 64 nombres indépendants ; elles ne savent pas que deux pixels sont voisins. C'est exactement ce que les réseaux de neurones dits « convolutifs » ajoutent. Avec 8 × 8 pixels et peu de scripteurs, l'algèbre linéaire fait pourtant presque aussi bien, avec des décisions qu'on peut expliquer.
- **Améliorer la robustesse.** On peut ajouter aux données d'apprentissage des copies décalées de chaque image : le modèle apprend alors que le décalage ne change pas le chiffre. Quel est le coût pour chaque méthode ?
- **Généraliser : le module.** Écrivez un fichier `reconnaissance.py` contenant une classe `Reconnaisseur(methode, k)` avec les méthodes `ajuster(X, y)`, `predire(X)`, `reussite(X, y)` et `predire_avec_rejet(X, seuil)`, et un fichier `test_reconnaissance.py` qui teste au moins : des classes bien séparées, l'équivalence entre sous-espaces avec $k = 0$ et chiffre moyen, l'indifférence à l'ordre des pixels, un point à égale distance de deux classes (rejet), et des entrées invalides. Utilisez-le pour mesurer l'effet de l'ajout de copies décalées à l'apprentissage. Le corrigé se trouve dans le dossier `corrige`.

In [ ]:
# À compléter : créez reconnaissance.py et test_reconnaissance.py, puis lancez vos tests et l'expérience des copies décalées

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
import importlib
tests = importlib.import_module("test_reconnaissance")
for nom in sorted(dir(tests)):
    if nom.startswith("test_"):
        getattr(tests, nom)()
        print("ok", nom)

decalages = [(0, 0), (1, 0), (-1, 0), (0, 1), (0, -1)]
X_aug = np.vstack([decaler(X_app, dx, dy) for dx, dy in decalages])
y_aug = np.tile(y_app, len(decalages))
print()
for nom, r in methodes.items():
    r2 = Reconnaisseur(r.methode, r.k).ajuster(X_aug, y_aug)
    print(f"{nom:22s} test {r2.reussite(X_test, y_test):.1%} (avant {r.reussite(X_test, y_test):.1%}) ; "
          f"décalé {r2.reussite(epreuves['décalage 1 px à droite'], y_test):.1%} "
          f"(avant {r.reussite(epreuves['décalage 1 px à droite'], y_test):.1%}) ; mémoire × {r2.taille_memoire() / r.taille_memoire():.1f}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Identifier erreurs systématiques et aléatoires": 1,
    "Implémenter un modèle général sous forme de programme robuste": 1,
    "Structurer un rapport ou une présentation": 1,
    "Expliquer oralement": 1,
    "Interpréter le travail des autres": 1,
    "Comprendre et critiquer les idées présentées": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "C4-evaluer-et-livrer", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")